In [65]:
import os
import pandas as pd
import numpy as np

all_samples = {}

# Directory where miner will save the RNA-seq output
output_folder = r"D:\School\IITD\General\GBM_new\rnaseq_output"

# Expression data was downloaded as a folder of folders
# Each subfolder has a file corresponding to it. The subfolder is given a Universally Unique Identifier (UUID)

for uuid in os.listdir(output_folder):
    uuid_path = os.path.join(output_folder, uuid)           # Loops through all folders and skips anything that isn't a subfolder
    
    if not os.path.isdir(uuid_path):
        continue
    
    tsv_files = [f for f in os.listdir(uuid_path) if f.endswith(".tsv")]     # List of all tsv filepaths in subfolders
    
    if not tsv_files:
        continue
    
    tsv_path = os.path.join(uuid_path, tsv_files[0])                         # [0] as each folder has only one file i.e len(tsv_files) == 1
    
    try:

        # Reads the TSV file into a dataframe
        df = pd.read_csv(tsv_path, sep="\t", comment="#", header=0) 
        
        # Skip summary rows (N_unmapped, N_multimapping etc. Only keeps actual genes i.e those with an Ensembl ID
        df = df[df["gene_id"].str.startswith("ENSG")]
        
        # Filter to protein-coding genes and drop all NaNs as done in the paper: "Gene expression RNA-seq data were TMM-normalized and Z-scored
        # using 10,263 genes after filtering"
        df = df[df["gene_type"] == "protein_coding"]    
        df = df[["gene_name", "tpm_unstranded"]].dropna()
         
        # Set index to gene name
        df = df.set_index("gene_name")
        
        # Use FOLDER name (which is the file UUID) not filename as the sample_id
        # dict: all_samples, its keys are UUIDs (corresponding to one file and in turn one sample). Sample MAY NOT BE A PRIMARY TUMOUR
        # Hence we filter later 
        # values are gene expression series (tpm_unstranded column of df) indexed by gene symbol or gene name
        all_samples[uuid] = df["tpm_unstranded"]
        
    except Exception as e:
        print(f"Error in {uuid}: {e}")

print(f"Loaded {len(all_samples)} samples")
tpm_matrix = pd.DataFrame(all_samples)
tpm_matrix = tpm_matrix.dropna()
#print(f"Matrix shape: {tpm_matrix.shape}")
#print(tpm_matrix.head())

# Loaded 391 samples
# Matrix shape: (19962, 391), excluding NaNs and non-coding genes

# Summary: tpm_matrix is a 19962 genes * 391 samples matrix
# The cells of the matrix contain TPM expression values of each gene in each sample



Loaded 391 samples
Matrix shape: (19962, 391)


In [ ]:
# Our current z-scored and tpm matrices contain UUID
# We need to map each UUID to its respective TCGA barcode since the rest of the data works on barcodes
# To do this we call the GDC API

uuids = tpm_matrix.columns.tolist()
uuid_to_barcode = {}
uuid_to_sample = {}
batch_size = 200

for i in range(0, len(uuids), batch_size):
    batch = uuids[i:i+batch_size]
    payload = {
        "filters": {
            "op": "in",
            "content": {"field": "file_id", "value": batch}
        },
        "fields": "file_id,cases.submitter_id,cases.samples.submitter_id,cases.samples.sample_type",
        "size": len(batch)
    }
    response = requests.post("https://api.gdc.cancer.gov/files", json=payload)
    
    for hit in response.json()["data"]["hits"]:
        file_id = hit["file_id"]
        case_id = hit["cases"][0]["submitter_id"]
        samples = hit["cases"][0].get("samples", [])
        uuid_to_barcode[file_id] = case_id
        uuid_to_sample[file_id] = [(s["submitter_id"], s["sample_type"]) for s in samples]
    
    print(f"Processed {min(i+batch_size, len(uuids))}/{len(uuids)}")

print(f"\nMapped: {len(uuid_to_barcode)}")

# Check sample types
all_sample_types = set()
for v in uuid_to_sample.values():
    for sample_id, sample_type in v:
        all_sample_types.add(sample_type)
print(f"Sample types found: {all_sample_types}")

# Show a few examples
print("\nExamples:")
for uuid, samples in list(uuid_to_sample.items())[:5]:
    print(f"{uuid_to_barcode[uuid]}: {samples}")

Processed 200/391


In [67]:
# Build mapping from UUID to TCGA barcode keeping only primary tumor samples
uuid_to_barcode_filtered = {}

for file_id, samples in uuid_to_sample.items():
    primary = [s for s in samples if s[1] == "Primary Tumor"]
    if primary:
        uuid_to_barcode_filtered[file_id] = uuid_to_barcode[file_id]

print(f"Primary tumor samples: {len(uuid_to_barcode_filtered)}")
print(f"Removed (recurrent/normal): {391 - len(uuid_to_barcode_filtered)}")

# Rename columns, drop non-primary samples
tpm_matrix.columns = [uuid_to_barcode_filtered.get(col, None) for col in tpm_matrix.columns]
tpm_matrix = tpm_matrix.loc[:, tpm_matrix.columns.notna()]
print(f"Shape after filtering: {tpm_matrix.shape}")

# 372 primary tumour samples out of 391
# Shape after filtering is 19962 *372

Primary tumor samples: 372
Removed (recurrent/normal): 19
Shape after filtering: (19962, 372)


In [70]:
dupes = tpm_matrix.columns[tpm_matrix.columns.duplicated(keep=False)].unique().tolist()
print(f"Duplicates: {len(dupes)}")

# 86 duplicates; these are handled by keeping only the sample w/ highest median expression

if dupes:
    non_dup = tpm_matrix.loc[:, ~tpm_matrix.columns.isin(dupes)]
    best_samples = []
    for patient in dupes:
        cols = tpm_matrix.loc[:, tpm_matrix.columns == patient]
        best = cols.iloc[:, cols.median(axis=0).argmax()]
        best_samples.append(best)
    best_dup = pd.concat(best_samples, axis=1)
    tpm_matrix = pd.concat([non_dup, best_dup], axis=1)

print(f"After dedup: {tpm_matrix.shape}")


Duplicates: 86
Step 4 - After dedup: (19962, 284)


In [73]:
# Converting to TPM and z-scoring as required by miner, then saving

from scipy import stats

log_tpm_matrix = np.log2(tpm_matrix + 1)
zscored_matrix = pd.DataFrame(
    stats.zscore(log_tpm_matrix, axis=1),
    index=log_tpm_matrix.index,
    columns=log_tpm_matrix.columns)

# Save
log_tpm_matrix.to_csv("rnaseq_log2tpm.csv")
zscored_matrix.to_csv("rnaseq_zscored.csv")
print("Saved.")

Saved.


In [76]:
# Final checks:

# tpm_matrix.shape() is (19962, 284)
# log2tpm size: 95.702581 MB
# zscored size: 109.535272 MB

In [117]:
# Now loading raw microarray data
micro = pd.read_csv(r"D:\School\IITD\General\GBM_new\gbm_tcga\data_mrna_affymetrix_microarray.txt", sep="\t", index_col=0)

# Step 2: Drop Entrez column
micro = micro.drop(columns=["Entrez_Gene_Id"])

# Step 3: Strip sample suffixes to get patient_ID as column and set index to gene name to match RNA-seq data
micro.columns = ["-".join(col.split("-")[:3]) for col in micro.columns]
micro.index.name = "gene_name"

print(f"Shape: {micro.shape}")

# Step 4: Check duplicates 
dup_genes = micro.index[micro.index.duplicated(keep=False)].unique()
print(f"Duplicate genes: {len(dup_genes)}")

# Count of duplicates, for future reference
for gene in dup_genes:
    print(f"  {gene}: {micro.index.tolist().count(gene)} times")

Shape: (12042, 528)
Duplicate genes: 12
  RBBP8: 2 times
  RREB1: 2 times
  KCNJ5: 2 times
  CD36: 2 times
  SNW1: 2 times
  MYLK: 2 times
  S100A10: 2 times
  RPS15: 2 times
  AGER: 2 times
  CX3CR1: 2 times
  REG1B: 2 times
  PTGDS: 2 times


In [124]:
# Testing: what does microarray data actually contain for a gene


gene = "RBBP8"
probes = micro.loc[gene]
#print(f"Type: {type(probes)}")
#print(f"Shape: {probes.shape}")
#print(probes.iloc[:, :3])

# There are multiple probes for 1 gene; in the paper, the probes were averaged (collapsed) if they mapped to the same gene and had correlated expression (>= 0.8)
# Basically if their values across all 528 patients had corr >= 0.8 they were considered to be measuring the same signal

# If not correlated, then the highest variance probe was kept, which is better when clustering for regulons

# Step 1: Collapse duplicates in microarray

dup_genes_micro = micro_raw.index[micro_raw.index.duplicated(keep=False)].unique()
print(f"Microarray duplicate genes: {len(dup_genes_micro)}")
non_dup_micro = micro_raw[~micro_raw.index.isin(dup_genes_micro)]
collapsed_micro = []

for gene in dup_genes_micro:
    probes = micro_raw.loc[gene].copy()
    row0 = probes.iloc[0]
    row1 = probes.iloc[1]
    corr_val = row0.corr(row1)
    if corr_val >= 0.8:
        result = probes.mean(axis=0)
        #print(f"{gene}: corr={corr_val:.2f} → averaged")
    else:
        result = row0 if row0.var() >= row1.var() else row1
        #print(f"{gene}: corr={corr_val:.2f} → kept highest variance probe")
    collapsed_micro.append(pd.DataFrame([result.values], index=[gene], columns=micro_raw.columns))

micro_clean = pd.concat([non_dup_micro, pd.concat(collapsed_micro)])
print(f"Microarray after collapsing: {micro_clean.shape}")
print(f"Duplicate genes remaining: {micro_clean.index.duplicated().sum()}")


# Step 2: Collapse duplicates in RNA-seq

dup_genes_rna = log_tpm_matrix.index[log_tpm_matrix.index.duplicated(keep=False)].unique()
print(f"\nRNA-seq duplicate genes: {len(dup_genes_rna)}")
non_dup_rna = log_tpm_matrix[~log_tpm_matrix.index.isin(dup_genes_rna)]
collapsed_rna = []

for gene in dup_genes_rna:
    probes = log_tpm_matrix.loc[gene].copy()
    row0 = probes.iloc[0]
    row1 = probes.iloc[1]
    corr_val = row0.corr(row1)
    if corr_val >= 0.8:
        result = probes.mean(axis=0)
        #print(f"{gene}: corr={corr_val:.2f} → averaged")
    else:
        result = row0 if row0.var() >= row1.var() else row1
        #print(f"{gene}: corr={corr_val:.2f} → kept highest variance probe")
    collapsed_rna.append(pd.DataFrame([result.values], index=[gene], columns=log_tpm_matrix.columns))

log_tpm_clean = pd.concat([non_dup_rna, pd.concat(collapsed_rna)])
print(f"RNA-seq after collapsing: {log_tpm_clean.shape}")
print(f"Duplicate genes remaining: {log_tpm_clean.index.duplicated().sum()}")

# Step 3: Filter to common genes
common_genes = micro_clean.index.intersection(log_tpm_clean.index)
print(f"\nCommon genes: {len(common_genes)}")

micro_common = micro_clean.loc[common_genes]
rnaseq_common = log_tpm_clean.loc[common_genes]

print(f"Microarray common: {micro_common.shape}")
print(f"RNA-seq common: {rnaseq_common.shape}")


Microarray duplicate genes: 12
Microarray after collapsing: (12030, 528)
Duplicate genes remaining: 0

RNA-seq duplicate genes: 24
RNA-seq after collapsing: (19938, 284)
Duplicate genes remaining: 0

Common genes: 10394
Microarray common: (10394, 528)
RNA-seq common: (10394, 284)


C:\Users\Amogh Kukreja\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\Amogh Kukreja\AppData\Roaming\Python\Python313\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [125]:
# Combine - prefer RNA-seq for overlapping patients
common_patients = micro_common.columns.intersection(rnaseq_common.columns)
rnaseq_only = rnaseq_common.columns.difference(micro_common.columns)
micro_only = micro_common.columns.difference(rnaseq_common.columns)

print(f"Patients in both: {len(common_patients)}")
print(f"RNA-seq only: {len(rnaseq_only)}")
print(f"Microarray only: {len(micro_only)}")
print(f"Total unique patients: {len(common_patients) + len(rnaseq_only) + len(micro_only)}")

combined_raw = pd.concat([
    micro_common.loc[:, micro_only],
    rnaseq_common.loc[:, rnaseq_only],
    rnaseq_common.loc[:, common_patients]
], axis=1)

print(f"\nCombined raw shape: {combined_raw.shape}")
print(f"Value range - Min: {combined_raw.min().min():.2f}, Max: {combined_raw.max().max():.2f}")

# Remove zero variance genes
zero_var = combined_raw.var(axis=1) == 0
print(f"Zero variance genes: {zero_var.sum()}")
combined_raw_filtered = combined_raw.loc[~zero_var]

# Z-score
combined_zscored = pd.DataFrame(
    stats.zscore(combined_raw_filtered.values, axis=1),
    index=combined_raw_filtered.index,
    columns=combined_raw_filtered.columns
)

'''

Final shape: (10394, 550)
Min: -6.89
Max: 8.11
Mean: 0.0000
NaN values: 0

'''

# Save
combined_raw_filtered.to_csv("combined_raw.csv")
combined_zscored.to_csv("combined_zscored.csv")
print("\nSaved combined_raw.csv and combined_zscored.csv")

Patients in both: 262
RNA-seq only: 22
Microarray only: 266
Total unique patients: 550

Combined raw shape: (10394, 550)
Value range - Min: 0.00, Max: 15.98
Zero variance genes: 0

Final shape: (10394, 550)
Min: -6.89
Max: 8.11
Mean: 0.0000
NaN values: 0

Saved combined_raw.csv and combined_zscored.csv


In [126]:
# 1. Shape check
print(f"Combined raw shape: {combined_raw_filtered.shape}")
print(f"Combined zscored shape: {combined_zscored.shape}")

# 2. No NaN or inf values
print(f"\nNaN in zscored: {combined_zscored.isna().sum().sum()}")
print(f"Inf in zscored: {np.isinf(combined_zscored.values).sum()}")

# 3. Z-score properties - mean should be ~0, std should be ~1 per gene
gene_means = combined_zscored.mean(axis=1)
gene_stds = combined_zscored.std(axis=1)
print(f"\nPer-gene mean - min:{gene_means.min():.4f} max:{gene_means.max():.4f} mean:{gene_means.mean():.4f}")
print(f"Per-gene std - min:{gene_stds.min():.4f} max:{gene_stds.max():.4f} mean:{gene_stds.mean():.4f}")

# 4. Check a known GBM gene - EGFR should be highly variable
print(f"\nEGFR stats:")
print(f"  mean: {combined_zscored.loc['EGFR'].mean():.4f}")
print(f"  std: {combined_zscored.loc['EGFR'].std():.4f}")
print(f"  min: {combined_zscored.loc['EGFR'].min():.4f}")
print(f"  max: {combined_zscored.loc['EGFR'].max():.4f}")

# 5. Check platform balance
micro_cols = set(micro_only.tolist())
rna_cols = set(rnaseq_only.tolist()) | set(common_patients.tolist())
print(f"\nMicroarray-only patients: {len(micro_only)}")
print(f"RNA-seq patients (overlap + rna-only): {len(common_patients) + len(rnaseq_only)}")

# 6. Check a known microarray-only and RNA-seq-only patient have valid values
micro_patient = micro_only[0]
rna_patient = rnaseq_only[0]
print(f"\nMicroarray patient {micro_patient} - NaN: {combined_zscored[micro_patient].isna().sum()}")
print(f"RNA-seq patient {rna_patient} - NaN: {combined_zscored[rna_patient].isna().sum()}")

Combined raw shape: (10394, 550)
Combined zscored shape: (10394, 550)

NaN in zscored: 0
Inf in zscored: 0

Per-gene mean - min:-0.0000 max:0.0000 mean:-0.0000
Per-gene std - min:1.0009 max:1.0009 mean:1.0009

EGFR stats:
  mean: -0.0000
  std: 1.0009
  min: -3.0601
  max: 2.2733

Microarray-only patients: 266
RNA-seq patients (overlap + rna-only): 284

Microarray patient TCGA-02-0001 - NaN: 0
RNA-seq patient TCGA-02-0055 - NaN: 0


In [ ]:
'''

Combined raw shape: (10394, 550)
Combined zscored shape: (10394, 550)

NaN in zscored: 0
Inf in zscored: 0

Per-gene mean - min:-0.0000 max:0.0000 mean:-0.0000
Per-gene std - min:1.0009 max:1.0009 mean:1.0009

EGFR stats:
  mean: -0.0000
  std: 1.0009
  min: -3.0601
  max: 2.2733

Microarray-only patients: 266
RNA-seq patients (overlap + rna-only): 284

Microarray patient TCGA-02-0001 - NaN: 0
RNA-seq patient TCGA-02-0055 - NaN: 0

'''